# Ingest the data

In [0]:
# define the path 
schema = "wines"
file_name = "wine_reviews-2026-04-11-23-44.csv"

file_path = f"/Volumes/workspace/default/{schema}/{file_name}"

df = spark.read.csv(file_path, header=True, inferSchema=True, multiLine=True, escape='"')

df.printSchema()

In [0]:
%skip 
import re

def clean_csv_newlines(src_path: str) -> str:
    dst_path = src_path.replace(".csv", "-cleaned.csv")
    with open(src_path, encoding="utf-8") as f:
        content = f.read()
    cleaned = re.sub(
        r'"([^"]*)"',
        lambda m: '"' + re.sub(r'[\r\n]+', ' ', m.group(1)) + '"',
        content,
        flags=re.DOTALL,
    )
    with open(dst_path, "w", encoding="utf-8", newline="") as f:
        f.write(cleaned)
    print(f"Cleaned CSV written to: {dst_path}")
    return dst_path
# no newlines in the reviews in csv now 
clean_path = clean_csv_newlines(file_path)


df = spark.read.csv(clean_path, header=True, inferSchema=True)

## Glimpse on data

In [0]:
print(f"Row count: {df.count():,}")

In [0]:
display(df.limit(10))

In [0]:
display(df)

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks data profile. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

## Save as Delta Bronze

In [0]:
df.write.format("delta").mode("overwrite").saveAsTable("wine_reviews_bronze")

print("Bronze table written.")

# Cleaning and casting 

## Casting column types

In [0]:
from pyspark.sql.functions import col, expr, regexp_replace, when, sum as spark_sum

cast_columns = [
    ("alcohol",         "double"),
    ("vintage",         "int"),
    ("case_production", "long"),
    ("retail",          "double"),
    ("rating",          "int"),
    ("bottle_size",     "double"),
]

# Build _cast and _error columns lazily — original columns kept intact for nulls_before
df_cast = df
for column, dtype in cast_columns:
    df_cast = (df_cast
        .withColumn(f"{column}_cast", expr(f"try_cast({column} as {dtype})"))
        .withColumn(f"{column}_error", when(
            col(f"{column}_cast").isNull() & col(column).isNotNull(), col(column)
        ))
    )

# Single agg pass: original column → nulls_before, _cast column → nulls_after
agg_exprs = []
for column, _ in cast_columns:
    agg_exprs.append(spark_sum(col(column).isNull().cast("int")).alias(f"{column}_nulls_before"))
    agg_exprs.append(spark_sum(col(f"{column}_cast").isNull().cast("int")).alias(f"{column}_nulls_after"))
    agg_exprs.append(spark_sum(col(f"{column}_error").isNotNull().cast("int")).alias(f"{column}_failures"))

stats = df_cast.agg(*agg_exprs).collect()[0]

# # Replace original columns with cast results now that stats are collected
# for column, _ in cast_columns:
#     df_cast = df_cast.withColumn(column, col(f"{column}_cast")).drop(f"{column}_cast")

# df_cast = df_cast.withColumn("is_nv", when(col("vintage_error") == "NV", 1).otherwise(0))

# print the stats
for column, _ in cast_columns:
    nulls_before = stats[f"{column}_nulls_before"] or 0
    nulls_after  = stats[f"{column}_nulls_after"]  or 0
    failures     = stats[f"{column}_failures"]      or 0
    print(f"{column:20s}  nulls before={nulls_before:>6,}  after={nulls_after:>6,}  failures={failures:>6,}")
    if failures > 0:
        df_cast.filter(col(f"{column}_error").isNotNull()).select(f"{column}_error").distinct().show(1000, truncate=False)


In [0]:
df_cast.printSchema()

## Special cleaning and feature creation

### alcohol

In [0]:
df_cast.groupBy("alcohol").count().orderBy("count", ascending=False).show(1000, truncate=False)

In [0]:
df_cast.groupBy("alcohol").count().orderBy("alcohol", ascending=False).show(1000, truncate=False)

In [0]:
df_cast = df_cast.withColumn(
    "alcohol",
    when(col("alcohol") > 25.0, None)
    .when((col("alcohol") <= 25.0) & (col("alcohol") > 7.0), col("alcohol") / 100)
    .when((col("alcohol") <= 7.0) & (col("alcohol") > 2.5), None)
    .when((col("alcohol") <= 2.5) & (col("alcohol") > 0.7), col("alcohol") / 10)
    .when((col("alcohol") <= 0.7) & (col("alcohol") > 0.25), None)
    .when((col("alcohol") <= 0.25) & (col("alcohol") >= 0.05), col("alcohol"))
    .when(col("alcohol") < 0.05, None)
    .otherwise(None)
)

In [0]:
df_cast.groupBy("alcohol").count().orderBy("count", ascending=False).show(1000, truncate=False)

### vintage

In [0]:
df_cast.select("vintage").distinct().count()

In [0]:
df_cast.groupBy("vintage").count().orderBy("count", ascending=False).show(1000, truncate=False)

In [0]:
# create a new feature column "is_nv" directly when casting vintage
df_cast = df_cast.withColumn("is_nv", when(col("vintage_error") == "NV", 1).otherwise(0))

### case_production

In [0]:
df_cast.select("case_production").distinct().count()

In [0]:
df_cast.groupBy("case_production").count().orderBy("count", ascending=False).show(10000, truncate=False)

### retail

In [0]:
df_cast.groupBy("retail").count().orderBy("count", ascending=False).show(1000, truncate=False)

In [0]:
df_cast = df_cast.withColumn(
    "retail",
    when(col("retail") == 0.0, None).
    otherwise(col("retail"))
)

### rating

In [0]:
df_cast.groupBy("rating").count().orderBy("count", ascending=False).show(1000, truncate=False)

In [0]:
df_cast.groupBy("rating").count().orderBy("rating", ascending=False).show(1000, truncate=False)

In [0]:
df_cast = df_cast.withColumn(
    "rating",
    when(col("rating") == 0.0, None).
    otherwise(col("rating"))
)

### bottle_size

In [0]:
df_cast.groupBy("bottle_size").count().orderBy("count", ascending=False).show(1000, truncate=False)

In [0]:
# Custom logic for bottle_size column - handle strings first, replacing ml (without any additional factor) and L with a *1000 multiplication factor before casting to double
# exception for 'L ml' 
df_cast = df_cast.withColumn(                                               
    "bottle_size",
    when(col("bottle_size").ilike("%L ml"),                                       
        expr("try_cast(trim(left(bottle_size, length(bottle_size) - 4)) as double)") * 1000).
    when(col("bottle_size").ilike("%ml"),                                       
        expr("try_cast(trim(left(bottle_size, length(bottle_size) - 2)) as double)")).
    when(col("bottle_size").ilike("%l"),  
        expr("try_cast(trim(left(bottle_size, length(bottle_size) - 1)) as double)") * 1000).
    otherwise(expr("try_cast(bottle_size as double)"))
)

In [0]:
valid_sizes = [750.0, 375.0, 500.0, 1500.0, 3000.0, 250.0, 1000.0, 187.0, 620.0]

df_cast = df_cast.withColumn(
    "bottle_size",
    when(col("bottle_size") == 75.0, 750.0).
    when(col("bottle_size") == 751.0, 750.0).
    when(col("bottle_size") == 740.0, 750.0).
    when(col("bottle_size").isin(valid_sizes), col("bottle_size"))
    .otherwise(None)
)

In [0]:
df_cast.groupBy("bottle_size").count().orderBy("count", ascending=False).show(1000, truncate=False)

## Summary of nulls

In [0]:
# get a summary of all null values in the original columns after casting, to identify any remaining issues or patterns in the data
from pyspark.sql.functions import col, sum as spark_sum
                                                                            
null_counts = df_cast.agg(*[                                              
    spark_sum(col(c).isNull().try_cast("int")).alias(c) for c in df.columns]).collect()[0]

print(f"{'column':<25}  {'nulls':>8}")
print("-" * 37)
for c in df.columns:
    n = null_counts[c] or 0
    print(f"{c:<25}  {n:>8,}")

In [0]:
display(df_cast)